In [3]:
#!/usr/bin/env python3
"""
Generate all parameter .in files for the beta / kCutOff / H0 scan.

Grid:
  beta    : 0.05, 0.1, 0.3, 1.0, 2.0   (lambda2 derived via beta = 3*lambda2/sqrt(8*lambda1))
  kCutOff : 1, 3, 5
  H0      : 0.3*mu, 1*mu, 3*mu

Total: 5 x 3 x 3 = 45 files  ->  src/models/parameter-files/scan/
"""

import os
import numpy as np

# ── Fixed parameters ──────────────────────────────────────────────────────────
MU       = 0.0547723   # GeV
LAMBDA1  = 0.003
N        = 512
LSIDE    = 100
DT       = 0.01
T_MAX    = 50
REMOTE_BASE = "/mt/user-batch/dpasari/scan_alt_tests"

# ── Sweep axes ────────────────────────────────────────────────────────────────
BETAS    = [ 0.1,  1.0]
KCUTOFFS = [ 3]
H0_GRID  = [
    ("0p3mu", round(0.3 * MU, 8)),   # ~0.0164 GeV
    ("1mu",   MU),                    # ~0.0548 GeV
]

# ── Helpers ───────────────────────────────────────────────────────────────────
def beta_str(b: float) -> str:
    """0.05 -> '0p05',0.3 -> '0p3',  1.0 -> '1',  2.0 -> '2'"""
    return f"{b:g}".replace(".", "p")

TEMPLATE = """\
#Output
outputfile = {results_dir}

#Evolution
expansion = true
evolver = LF

#Lattice
N = {N}
dt = {dt}
lSide = {lSide}

#Times
t0 = 0
tOutputFreq  = 0.1
tOutputInfreq  = 5
tOutputRareFreq = 3
tMax = {tmax}

#Spectra options
PS_type = 1
PS_version = 1

#GWs
GWprojectorType = 2
withGWs = false

fixedBackground = true
omegaEoS = 0.3333
H0 = {H0:.8f}   # {H0_ratio:.4f} * mu

#IC
kCutOff = {kcut}
initial_amplitudes = 0 0
initial_momenta    = 0 0

#Model Parameters
mu      = {mu}
lambda1 = {lambda1}
lambda2 = {lambda2:.8f}   # beta = {beta}
"""

# ── Generate files ────────────────────────────────────────────────────────────
# __file__ is not defined in Jupyter notebooks, so fall back to cwd
try:
    _root = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _root = os.getcwd()   # cwd should be the CosmoLattice_Zn project root

out_dir = os.path.join(_root,"../" "src", "models", "parameter-files", "scan_alt_tests")
os.makedirs(out_dir, exist_ok=True)

generated = []
for beta in BETAS:
    lambda2 = beta * np.sqrt(8 * LAMBDA1) / 3
    bs = beta_str(beta)

    for kcut in KCUTOFFS:
        for h_label, H0 in H0_GRID:
            fname   = f"DWZ3_b{bs}_H{h_label}_k{kcut}.in"
            res_dir = f"{REMOTE_BASE}/results_z3_b{bs}_H{h_label}_k{kcut}_alt_tests/"

            content = TEMPLATE.format(
                results_dir = res_dir,
                N           = N,
                dt          = DT,
                lSide       = LSIDE,
                tmax        = T_MAX,
                H0          = H0,
                H0_ratio    = H0 / MU,
                kcut        = kcut,
                mu          = MU,
                lambda1     = LAMBDA1,
                lambda2     = lambda2,
                beta        = beta,
            )

            fpath = os.path.join(out_dir, fname)
            with open(fpath, "w") as fh:
                fh.write(content)
            generated.append(fname)

print(f"Generated {len(generated)} files -> {out_dir}")
for f in generated:
    print(f"  {f}")

            # print( res_dir)


Generated 4 files -> /mt/home/dpasari/CosmoLattice_Zn/analysis/../src/models/parameter-files/scan_alt_tests
  DWZ3_b0p1_H0p3mu_k3.in
  DWZ3_b0p1_H1mu_k3.in
  DWZ3_b1_H0p3mu_k3.in
  DWZ3_b1_H1mu_k3.in
